In [1]:
# CELL 0: Background keepalive (prevents Colab timeout)
import threading, time

def _keepalive():
    while True:
        time.sleep(55)
        _ = 1 + 1

if not any(t.name == 'keepalive' for t in threading.enumerate()):
    t = threading.Thread(target=_keepalive, name='keepalive', daemon=True)
    t.start()
    print('Keepalive thread started')
else:
    print('Keepalive already running')

Keepalive thread started


In [2]:
# CELL 1: Mount Google Drive
import os, shutil, time
from google.colab import drive

MOUNT = '/content/drive'

if os.path.exists(MOUNT):
    try:
        shutil.rmtree(MOUNT, ignore_errors=True)
    except:
        pass

for attempt in range(1, 6):
    try:
        drive.mount(MOUNT, force_remount=True)
        print('Drive mounted!')
        break
    except Exception as e:
        if attempt < 5:
            wait = 10 * attempt
            print(f'Mount {attempt}/5 failed: {e}, retry in {wait}s')
            time.sleep(wait)
        else:
            raise

Mounted at /content/drive
Drive mounted!


In [3]:
# CELL 2: Config & Load CSV
# ─── YOUR DRIVE PATHS ─────────────────────────────────────────────────────────
CSV_PATH   = '/content/drive/MyDrive/ArthLLM-2B/section 3 (bonus split)/nse_bonus_split.csv'
OUTPUT_DIR = '/content/drive/MyDrive/ArthLLM-2B/section 3 (bonus split)/pdfs'
# ──────────────────────────────────────────────────────────────────────────────

import os, pandas as pd

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f'CSV not found: {CSV_PATH}\nMake sure you uploaded it to the correct Drive folder.')

df = pd.read_csv(CSV_PATH, dtype=str, on_bad_lines='skip', engine='python')
print(f'Total rows     : {len(df)}')
print(f'Columns        : {list(df.columns)}')

ann = df[df['source'] == 'ANNOUNCEMENT'].copy()
ann = ann[ann['pdf_url'].notna() & (ann['pdf_url'].str.strip() != '')]
print(f'Downloadable rows (ANNOUNCEMENT with pdf_url): {len(ann)}')

if 'download_status' not in df.columns:
    df['download_status'] = ''
    df.to_csv(CSV_PATH, index=False, quoting=1)
    print('Added download_status column and saved CSV')

print(f'\nCSV  → {CSV_PATH}')
print(f'PDFs → {OUTPUT_DIR}')

Total rows     : 4042
Columns        : ['source', 'symbol', 'company', 'series', 'isin', 'industry', 'subject', 'action_type', 'ratio', 'ex_date', 'record_date', 'bc_start_date', 'bc_end_date', 'face_value', 'broadcast_date', 'seq_id', 'filing_date', 'sort_date', 'desc', 'announcement_text', 'pdf_url', 'file_size', 'status']
Downloadable rows (ANNOUNCEMENT with pdf_url): 2945
Added download_status column and saved CSV

CSV  → /content/drive/MyDrive/ArthLLM-2B/section 3 (bonus split)/nse_bonus_split.csv
PDFs → /content/drive/MyDrive/ArthLLM-2B/section 3 (bonus split)/pdfs


In [4]:
# CELL 3: Sync manifest with files already on Drive (safe to re-run)
import os, pandas as pd

df = pd.read_csv(CSV_PATH, dtype=str, on_bad_lines='skip', engine='python')
if 'download_status' not in df.columns:
    df['download_status'] = ''

ann_mask = (
    (df['source'] == 'ANNOUNCEMENT') &
    df['pdf_url'].notna() &
    (df['pdf_url'].str.strip() != '')
)

def make_filename(row):
    symbol = ''.join(c for c in str(row.get('symbol','UNKNOWN')).strip() if c.isalnum() or c in '-_')[:20]
    seq_id = ''.join(c for c in str(row.get('seq_id','')).strip() if c.isalnum() or c in '-_')
    action = ''.join(c for c in str(row.get('action_type','X')).strip() if c.isalnum() or c in '-_')
    sort_d = str(row.get('sort_date','')).strip().replace(' ','_').replace(':','-')[:10]
    return f'{sort_d}_{symbol}_{action}_{seq_id}.pdf'

print('Scanning Drive for existing PDFs...')
existing = set(os.listdir(OUTPUT_DIR))
print(f'Found {len(existing)} PDFs already on Drive')

updated = 0
for idx in df[ann_mask].index:
    row = df.loc[idx]
    fn  = make_filename(row)
    if fn in existing and df.at[idx, 'download_status'] != 'downloaded':
        df.at[idx, 'download_status'] = 'downloaded'
        updated += 1

if updated:
    df.to_csv(CSV_PATH, index=False, quoting=1)
    print(f'Marked {updated} rows as already downloaded')

done = (df.loc[ann_mask, 'download_status'] == 'downloaded').sum()
pend = ann_mask.sum() - done
print(f'\nTotal={ann_mask.sum()} | Done={done} | Pending={pend}')
print('Ready for Cell 4.')

Scanning Drive for existing PDFs...
Found 0 PDFs already on Drive

Total=2945 | Done=0 | Pending=2945
Ready for Cell 4.


In [6]:
# CELL 4: Download PDFs — NSE cookie handshake fix
import os, time, random, threading, concurrent.futures, requests
import pandas as pd
from tqdm.notebook import tqdm

# ── Knobs ──────────────────────────────────────────────────────────────────────
MAX_WORKERS = 8
SAVE_EVERY  = 25
TIMER_SAVE  = 120
TIMEOUT     = 45
MAX_RETRIES = 3
# ──────────────────────────────────────────────────────────────────────────────

thread_local = threading.local()
write_lock   = threading.Lock()
stats_lock   = threading.Lock()
stats        = {'success': 0, 'skipped': 0, 'failed': 0, 'invalid': 0}
save_counter = 0

NSE_HEADERS = {
    'User-Agent'     : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept'         : 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection'     : 'keep-alive',
    'DNT'            : '1',
}

def warm_up_session(s):
    """Visit NSE homepage + archives to get valid cookies — critical for PDF access."""
    try:
        s.get('https://www.nseindia.com/', headers=NSE_HEADERS, timeout=15)
        time.sleep(1.5)
        s.get('https://nsearchives.nseindia.com/', headers=NSE_HEADERS, timeout=15)
        time.sleep(1.0)
    except Exception as e:
        print(f'[warm-up warning] {e}')

def get_session():
    if not hasattr(thread_local, 'session'):
        s = requests.Session()
        s.headers.update(NSE_HEADERS)
        warm_up_session(s)
        thread_local.session = s
        thread_local.request_count = 0
    return thread_local.session

def refresh_session_if_needed():
    """Re-warm cookies every 200 requests per thread to avoid expiry."""
    thread_local.request_count = getattr(thread_local, 'request_count', 0) + 1
    if thread_local.request_count % 200 == 0:
        warm_up_session(thread_local.session)

def safe_str(v):
    return ''.join(c for c in str(v).strip() if c.isalnum() or c in '-_') if pd.notna(v) else ''

def make_filename(row):
    symbol = safe_str(row.get('symbol', 'UNKNOWN'))[:20]
    seq_id = safe_str(row.get('seq_id', ''))
    action = safe_str(row.get('action_type', 'X'))
    sort_d = str(row.get('sort_date', '')).strip().replace(' ', '_').replace(':', '-')[:10]
    return f'{sort_d}_{symbol}_{action}_{seq_id}.pdf'

def download_one(row):
    global save_counter
    idx = row['_idx']
    url = str(row.get('pdf_url', '')).strip()

    if not url.startswith('http'):
        return idx, 'invalid'

    fn   = make_filename(row)
    path = os.path.join(OUTPUT_DIR, fn)

    if os.path.exists(path) and os.path.getsize(path) > 1024:
        with stats_lock:
            stats['skipped'] += 1
        return idx, 'downloaded'

    session = get_session()
    refresh_session_if_needed()

    # Use pdf-specific headers for actual download request
    pdf_headers = {
        **NSE_HEADERS,
        'Accept'  : 'application/pdf,application/octet-stream,*/*;q=0.8',
        'Referer' : 'https://www.nseindia.com/',
    }

    time.sleep(0.5 + random.uniform(0, 0.5))
    result = 'failed'

    for attempt in range(MAX_RETRIES):
        try:
            r = session.get(url, headers=pdf_headers, timeout=TIMEOUT)

            if r.status_code == 429:
                time.sleep(60 + random.uniform(0, 30))
                continue
            if r.status_code == 503:
                time.sleep(30)
                continue
            if r.status_code in (403, 404, 400):
                result = 'failed'
                break

            if r.status_code == 200:
                content = r.content

                # Check content type header first
                ct = r.headers.get('Content-Type', '')
                is_pdf_header = 'pdf' in ct.lower() or 'octet-stream' in ct.lower()

                # Check magic bytes
                is_pdf_bytes = content.lstrip().startswith(b'%PDF')

                if not is_pdf_bytes:
                    # NSE sometimes returns HTML error page — re-warm and retry
                    if attempt < MAX_RETRIES - 1:
                        warm_up_session(session)
                        time.sleep(3 + attempt * 2)
                        continue
                    else:
                        result = 'invalid'
                        break

                # Valid PDF — write atomically
                tmp = path + '.tmp'
                with open(tmp, 'wb') as f:
                    f.write(content)
                os.rename(tmp, path)
                result = 'downloaded'
                with stats_lock:
                    stats['success'] += 1
                break

        except requests.exceptions.Timeout:
            if attempt == MAX_RETRIES - 1:
                result = 'failed'
            time.sleep(5)
        except Exception:
            if attempt == MAX_RETRIES - 1:
                result = 'failed'
            time.sleep(3)

    if result == 'invalid':
        with stats_lock:
            stats['invalid'] += 1
    elif result == 'failed':
        with stats_lock:
            stats['failed'] += 1

    return idx, result

# ── Load manifest ──────────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, dtype=str, on_bad_lines='skip', engine='python')
if 'download_status' not in df.columns:
    df['download_status'] = ''
df = df.reset_index().rename(columns={'index': '_idx'})

ann_mask = (
    (df['source'] == 'ANNOUNCEMENT') &
    df['pdf_url'].notna() &
    (df['pdf_url'].str.strip() != '')
)

# Reset previously 'invalid' rows — they were likely just blocked, not truly invalid
invalid_count = (df['download_status'] == 'invalid').sum()
if invalid_count > 0:
    df.loc[df['download_status'] == 'invalid', 'download_status'] = ''
    print(f'Reset {invalid_count} invalid → pending (will retry with proper cookies)')

# Pre-flight disk check
existing = set(os.listdir(OUTPUT_DIR))
pre = 0
for idx in df[ann_mask & (df['download_status'] != 'downloaded')].index:
    if make_filename(df.loc[idx]) in existing:
        df.at[idx, 'download_status'] = 'downloaded'
        pre += 1
if pre:
    print(f'Pre-flight: {pre} already on disk')

todo       = df[ann_mask & (df['download_status'] != 'downloaded')].to_dict('records')
total      = len(todo)
total_done = ann_mask.sum() - total
print(f'Total={ann_mask.sum()} | Done={total_done} | To download={total}')
print(f'Warming up {MAX_WORKERS} NSE sessions (takes ~{MAX_WORKERS*3}s)...')

# ── Autosave ───────────────────────────────────────────────────────────────────
_stop = threading.Event()
def _autosave():
    while not _stop.wait(TIMER_SAVE):
        with write_lock:
            try:
                df.drop(columns=['_idx']).to_csv(CSV_PATH, index=False, quoting=1)
            except Exception:
                pass
threading.Thread(target=_autosave, daemon=True, name='autosave').start()

# ── Download ───────────────────────────────────────────────────────────────────
try:
    with tqdm(total=total, desc='Downloading', unit='pdf', colour='green') as pbar:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {ex.submit(download_one, r): r for r in todo}
            for fut in concurrent.futures.as_completed(futs):
                idx, st = fut.result()
                df.loc[df['_idx'] == idx, 'download_status'] = st
                pbar.update(1)
                save_counter += 1
                if save_counter >= SAVE_EVERY:
                    with write_lock:
                        df.drop(columns=['_idx']).to_csv(CSV_PATH, index=False, quoting=1)
                    save_counter = 0
                pbar.set_postfix(ok=stats['success'], skip=stats['skipped'],
                                 fail=stats['failed'], inv=stats['invalid'], refresh=False)
except KeyboardInterrupt:
    print('\nInterrupted — saving...')
finally:
    _stop.set()
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith('.tmp'):
            try: os.remove(os.path.join(OUTPUT_DIR, f))
            except: pass
    with write_lock:
        df.drop(columns=['_idx']).to_csv(CSV_PATH, index=False, quoting=1)
    print(f'Saved → {CSV_PATH}')

print(f"\n✅ success={stats['success']}  ⏭ skip={stats['skipped']}  "
      f"❌ fail={stats['failed']}  ⚠ inv={stats['invalid']}")

Reset 1106 invalid → pending (will retry with proper cookies)
Pre-flight: 177 already on disk
Total=2945 | Done=1071 | To download=1874
Warming up 8 NSE sessions (takes ~24s)...


Downloading:   0%|          | 0/1874 [00:00<?, ?pdf/s]

Saved → /content/drive/MyDrive/ArthLLM-2B/section 3 (bonus split)/nse_bonus_split.csv

✅ success=771  ⏭ skip=764  ❌ fail=1  ⚠ inv=772


In [ ]:
# CELL 5 (optional): Reset failed rows → re-run Cell 4 to retry them
df = pd.read_csv(CSV_PATH, dtype=str, on_bad_lines='skip', engine='python')
failed = (df['download_status'] == 'failed').sum()
print(f'Failed rows: {failed}')
df.loc[df['download_status'] == 'failed', 'download_status'] = ''
df.to_csv(CSV_PATH, index=False, quoting=1)
print('Reset to pending. Now re-run Cell 4.')